In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
source = pd.read_csv('/content/drive/MyDrive/파일위치/pan_qna_kkm_w2v.csv')
source.head()

,question,answer,embedding
0,개인 금융 기관 자금 대출 걸 나 대출금 상환 일련 사법 상 계약 아야,"행정심판법 제2조 제1항 및 제3조 제1항의 규정에 의하면, 행정심판은 행정청이 행...",[ 0.03622955 0.23117849 1.4854755 -0.748379...
1,종전 규정 허가 자 신 규정 적용 매립 목적 변경 대상 알 니 라는 이유 행하 부당,기타 주변여건의 변화 등으로 인하여 매립목적의 변경이 불가피한 경우인지 등을 종합적...,[ 0.86184657 -0.0624576 1.3320264 -0.901639...
2,군 복무 중 질병 발병 악화 공상 판정 전역 이에 위원 회의 심의 도 군복무 중 걸...,청구인이 군 복무중 이 건 질병들이 발병 또는 악화되어 육군에서 공상판정을 받고 전...,[ 0.00594179 -0.7188672 0.0817882 -0.570121...
3,토지 임차인 지상 물 매수 청구권 배제 기로 하여 임차인 불리 약정 효력 다,임차인이 임대차계약을 체결할 당시 건물 기타 지상시설 일체를 포기하기로 하였다 해도...,[ 0.08671884 -0.67837673 1.4282916 -0.828758...
4,낙뢰 위험 상당 정도 예상 체육 시설 업자 골프장 운영자 이용자 피난 지시 내리 주...,체육시설업자로서는 낙뢰의 위험이 상당한 정도로 예상되는 경우에는 이용자에 대하여 피...,[ 0.47768384 -1.628598 0.2038537 -1.21344 ...


In [ ]:
def str_to_vector(emd):
    res = emd.split(' ')
    #print(emd)
    #print(len(r))
    r = []
    for a in res:
        if len(a) >2:
            r.append(a)
    if len(r) != 100:
        print(len(r))
    for i in range(len(r)):
        r[i] = r[i].replace('[', '')
        r[i] = r[i].replace(']', '')
        r[i] = r[i].replace('\n', '')
        r[i] = float(r[i])
    return np.array(r)

In [ ]:
import numpy as np
source['emd'] = source.apply(lambda x: str_to_vector(x['embedding']), axis=1)
emd2 = source['emd']
answer = source['answer']

In [ ]:
from numpy import dot
from numpy.linalg import norm
def cos_sim(A, B):
  return dot(A, B)/(norm(A)*norm(B))

In [ ]:
!pip install konlpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.1/494.1 kB 23.9 MB/s eta 0:00:00


In [ ]:
from konlpy.tag import Kkma

kkma = Kkma()

In [ ]:
from tqdm import tqdm
import re
stopwords = ['의','가','이','은','들','는','좀','잘','과','도','를','으로','자','에','와','한','하다',
             '적극', '소극','여부', '되다', '제', '매', '로', '때', '후', '로', '전', '민법', '방법',
             '경우', '상', '따르다', '있다', '않다', '원심', '및', '법', '에서', '또는', '그', '수', '에게',
             '인지', '해당', '에게', '위', '판결', '조', '인', '위', '사례', '사안', '대하', '되어다'
             '효력', '판단', '청구', '소송', '법원', '제기', '인정', '의미', '요건', '받다', '취지',
             '는지', '관하', '다고']


def tokenize_sentence(sentence):
    tokenized_sentence = []
    tokenized_sentence = kkma.morphs(sentence) # 토큰화
    stopwords_removed_sentence = [word for word in tokenized_sentence if not word in stopwords] # 불용어 제거
    l = ''
    for s in stopwords_removed_sentence:
        s = re.sub(r"[^가-힣\s]", " ", s)
        l += s +' '
    return l

In [ ]:
from gensim.models import Word2Vec

loaded_model = Word2Vec.load('/content/drive/MyDrive/파일위치/book_law_kkm_w2v.model')

In [ ]:
#임베딩하는 코드
##시용자 입력이 들어오면 (들어올 때마다) 사용자 입력과 전체 질문리스트(혹은 해당 카테고리의 질문 리스트)를 임베딩해줌
def get_document_vectors(document_list):
    document_embedding_list = []
    #각 문서에 대해서
    for line in document_list:
        doc2vec = None
        count = 0
        print(line)
        for word in line.split():
            print(word)
            try:
                w = loaded_model.wv[word]
                count += 1

                #해당 문서에 있는 모든 단어들의 벡터 값을 더한다.
                if doc2vec is None :
                    doc2vec = w
                else :
                    doc2vec = doc2vec + w
            except:
                pass

        if doc2vec is not None:
            # 단어 벡터를 모두 더한 벡터의 값을 문서 길이로 나눠준다.
            doc2vec = doc2vec/count
            document_embedding_list.append(doc2vec)
        else:
            document_embedding_list.append(None)
    # 각 문서에 대한 문서 벡터 리스트를 리턴
    return document_embedding_list


In [ ]:
def return_answer(question):
    q = tokenize_sentence(question)
    embedding = get_document_vectors([q])[0]

    #유사도 계산 및 정렬
    sim_scores = [[k, cos_sim(emd2[k], embedding)] for k in range(len(source))]
    sim_scores.sort(key=lambda x: x[1], reverse=True) #sim_scores의 각 리스트 중 두번째 요소를 정렬 기준으로
    #print(sim_scores[:10])

    #상위 다섯 개 저장(중복 제거)
    result = []
    for sim in sim_scores:
        a = answer[sim[0]]
        if a not in result:
            result.append(a)
        else:
            #print(a)
            pass
        if len(result) == 5:
            break
    return result

In [ ]:
return_answer('사실을 입력해도 명예훼손죄가 성립되는지?')

사실 을 입력 하 어도 명예 훼손죄 성립 되   
사실
을
입력
하
어도
명예
훼손죄
성립
되


['형법 제307조는 명예훼손죄를 규정하며, 제1항은 사실의 적시로 명예를 훼손한 경우, 제2항은 허위 사실의 적시로 명예를 훼손한 경우를 다룹니다. 제1항은 진실, 허위 여부와 무관하게 적용되며, 허위 사실이나 행위자가 그 허위성을 인식하지 못한 경우에도 제1항이 적용됩니다.',
 '명예훼손죄의 사실의 적시란 가치판단이나 평가를 내용으로 하는 의견표현에 대치되는 개념으로서 시간과 공간적으로 구체적인 과거 또는 현재의 사실관계에 관한 보고 내지 진술을 의미하는 것이며 그 표현내용이 증거에 의한 입증이 가능한 것을 말하고 그 표현 행위가 객관적으로 상대방의 인격적 가치에 대한 사회적 평가를 저하할 만했는지를 기준으로 판단하여야 한다.',
 '피고인은 비방할 목적으로 피해자가 거짓으로 결별 기사를 낸 뒤 이를 마케팅에 이용한 것인 양 적시하거나, 피해자가 탈세로 처벌될 것이라는 취지의 허위 사실을 적시하여 별지 범죄일람표 2 기재와 같이 2020. 2. 24.경까지 총 31개의 댓글을 작성하여 허위 사실을 적시하여 피해자의 명예를 훼손하였다.',
 '구 정보통신망 이용촉진 및 정보보호 등에 관한 법률 제61조 제2항의 정보통신망을 통한 허위사실 적시에 의한 명예훼손죄, 형법 제307조 제2항의 허위사실 적시에 의한 명예훼손죄가 성립하려면 그 적시하는 사실이 허위이어야 할 뿐 아니라, 피고인이 그와 같은 사실을 적시할 때에 적시사실이 허위임을 인식하여야 한다.',
 '형법 제307조 제2항의 허위사실 적시에 의한 명예훼손죄에서 적시된 사실이 허위인지 여부를 판단할 때 전체 취지를 살펴볼 때 중요한 부분이 진실한 경우 세부적 내용의 약소한 차이를 가지고 허위라고 볼 수 없다. 행위자의 허위성 인식 여부는 여러 객관적 사정을 종합하여 판단하며, 위 죄는 미필적 고의에 의하여도 성립하고, 이는 사자명예훼손죄의 판단에서도 마찬가지이다.']